# 打開黑盒子：看語言模型如何預測「下一個字」

---

## 這份程式要做什麼

前面的程式都是呼叫 `model.generate(...)` 或 `pipeline(...)`，一次就生成一整段文字。這一份**把 `generate` 裡面最核心的一步單獨拆出來**：

> 給模型一句話，看它**一步一步**算出「下一個字是什麼」。

```
提示詞： "The capital of France is"
              │ 分詞
              ▼
        token id 序列（N 個）
              │ ① model.model(...)         ← Transformer 本體
              ▼
        每個位置一個「隱藏向量」        形狀 (1, N, 隱藏維度)
              │ ② model.lm_head(...)       ← 語言模型的輸出頭
              ▼
        每個位置對「詞彙表每個字」的分數   形狀 (1, N, 詞彙表大小)
              │ ③ 取最後一個位置
              ▼
        詞彙表每個字的分數                形狀 (詞彙表大小,)
              │ ④ argmax：選分數最高的
              ▼
        一個 token id
              │ ⑤ decode
              ▼
        下一個字（預期是 "Paris"）
```

這個過程，就是語言模型「預測下一個詞」的全部。**生成一整段文字，只是把這個過程重複很多次**。

## 為什麼要學這個

| | 前面的程式 | **這份** |
|---|---|---|
| 做法 | `generate` / `pipeline` 一次生成很多字 | **只做一步，看清楚每個中間結果** |
| 學到什麼 | 怎麼使用模型 | **模型內部在做什麼** |

理解這一步，之後看到「logits」「隱藏狀態」「取樣」「溫度」這些名詞就不會陌生。

## 語言模型的兩大部分

```
CausalLM 模型  =  ① 本體（Transformer）  +  ② 語言模型輸出頭（lm_head）
                    model.model              model.lm_head
```

| 部分 | 做什麼 | 輸出 |
|---|---|---|
| ① 本體 `model.model` | 讀懂整段文字，把每個位置整理成一個「包含上下文的向量」 | 隱藏狀態（hidden states） |
| ② 輸出頭 `model.lm_head` | 把這個向量轉成「詞彙表裡每個字的分數」 | logits |

你可以把它想成：**本體負責「理解」，輸出頭負責「決定說哪個字」**。

## 步驟 0：載入套件

| 匯入 | 用途 |
|---|---|
| `torch` | PyTorch（這份程式裡**沒有直接用到**，但模型的張量運算都建立在它上面） |
| `AutoTokenizer` | 自動載入配對的分詞器 |
| `AutoModelForCausalLM` | 自動載入因果語言模型（本體 + lm_head） |
| `pipeline` | 現成的生成工具（這份程式建立了，但**後面沒有使用**） |

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

## 步驟 1：載入模型與分詞器

| 部分 | 說明 |
|---|---|
| `model_name` | Phi-3-mini 指令模型。存成變數，讓分詞器與模型用**同一個名稱**，確保配對 |
| `device_map="cuda"` | 把整個模型放到 GPU |
| `torch_dtype="auto"` | 依模型設定檔記錄的精度載入（通常是 16 位元），比預設的 32 位元省一半記憶體 |


In [ ]:
#載入模型與分詞器
model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",  
    torch_dtype="auto",
)

## 步驟 2：建立生成管道（這份程式沒有用到）

```python
generator = pipeline("text-generation", model=model, tokenizer=tokenizer,
                     return_full_text=False, max_new_tokens=50, do_sample=False)
```

這是上一份程式留下來的。**後面沒有任何地方呼叫 `generator`**，所以它不會影響結果。它的角色可以當作對照：

| | `generator(prompt)` | 這份程式後面做的事 |
|---|---|---|
| 做的事 | 重複預測下一個字，直到生成 50 個新 token（或遇到結束符號） | **只預測一次**，得到下一個字 |
| 結果 | 一整段文字 | 一個字（預期是 "Paris"） |

`pipeline` 本質上就是：**把後面這份程式做的事，重複 50 次**，每次把新預測的字接到輸入後面。

參數的意思（回顧）：`return_full_text=False` 只回傳新生成的部分；`max_new_tokens=50` 最多生成 50 個新 token；`do_sample=False` 每步都選機率最高的字（貪婪解碼）。

In [ ]:
#建立生成管道
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,  #只回傳結果 不包含提示詞
    max_new_tokens=50,
    do_sample=False,  #模型是否使用採樣策略選擇下一個字元
)

## 步驟 3：準備提示詞並分詞

```python
prompt = "The capital of France is"
```

這是一個「**句子沒寫完**」的提示詞，答案很明確：下一個字應該是 "Paris"。用這種句子觀察模型的預測，最容易驗證對不對。

```python
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
```

| 部分 | 說明 |
|---|---|
| `tokenizer(prompt, ...)` | 把文字切成 token 並轉成數字 |
| `return_tensors="pt"` | 回傳 PyTorch 張量 |
| `.input_ids` | 取出 token id |
| `.to("cuda")` | 搬到 GPU，要和模型在同一個裝置 |

`input_ids` 的形狀是 `(1, N)`：1 句話、N 個 token。這句話被切成約 6 個 token（我依 Llama 系列分詞器的常見行為估計：開頭的特殊符號 + The / capital / of / France / is，沒有實際印出，建議 `print(input_ids.shape)` 確認）。

In [ ]:
#定義提示詞
prompt = "The capital of France is"

In [ ]:
#將提示詞轉換為模型輸入格式(分詞)
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

## 步驟 4：通過 Transformer 本體 —— 得到「隱藏狀態」

```python
model_output = model.model(input_ids=input_ids)
```

### `model.model` 是什麼

`model` 是完整的因果語言模型，它**裡面有一個叫 `model` 的屬性，就是 Transformer 本體**，所以才會寫成 `model.model`：

```
model                      ← AutoModelForCausalLM（完整模型）
 ├── model.model           ← Transformer 本體（詞嵌入 → 很多層 Transformer → 最後的正規化）
 └── model.lm_head         ← 語言模型輸出頭
```

### 它做了什麼

```
N 個 token id  →  詞嵌入  →  一層一層的 Transformer（每層讓每個位置「看」前面的字）  →  每個位置一個向量
```

經過很多層之後，**每個位置的向量都已經吸收了它前面所有字的資訊**。例如最後一個位置（`is`）的向量，包含了「The capital of France is」整句的語意。

### 輸出的形狀

`model_output` 是一個物件，裡面最重要的是**最後一層的隱藏狀態**，可以用 `model_output[0]` 取出（下一步會用到）：

```
model_output[0] 的形狀：(1, N, 隱藏維度)
```

| 維度 | 意思 |
|---|---|
| `1` | 一句話（batch 大小） |
| `N` | N 個 token，**每個位置各有一個向量** |
| 隱藏維度 | 每個向量有幾個數字。我印象中 Phi-3-mini 是 3072（沒有確認你的版本） |

### ⚠️ 記憶體小提醒：沒有 `torch.no_grad()`

PyTorch 預設會在前向傳播時**記錄計算圖**（為了之後反向傳播訓練用）。這份程式只是在「使用」模型，不需要這些記錄，卻沒有關掉，會多佔一些顯存。對 3.8B 的模型、8 GB 的顯卡來說，這可能是壓垮駱駝的最後一根稻草。

通常的寫法是用 `with torch.no_grad():` 包起來（你有 `import torch`，應該就是為了這個，見實驗區）。我沒有實測你的環境。

In [ ]:
#lm前的輸出
model_output = model.model(input_ids=input_ids)

## 步驟 5：通過語言模型輸出頭 —— 得到「每個字的分數」

```python
lm_model_output = model.lm_head(model_output[0])
```

### `lm_head` 是什麼

`lm_head`（language modeling head）是**一個線性層**：

```
輸入：  每個位置的隱藏向量       （隱藏維度，例如 3072 個數字）
輸出：  詞彙表裡「每一個字」的分數 （詞彙表大小，例如 3 萬多個數字）
```

簡單說：它是一個「**把向量換算成各個字的得分**」的矩陣乘法。Phi-3-mini 的詞彙表大小約 32064。

### 輸出的形狀

```
lm_model_output 的形狀：(1, N, 詞彙表大小)
```

**每個位置都有一組分數**，每組有詞彙表大小那麼多個數字。這些分數叫 **logits**。

### logits 不是機率

| | logits | 機率 |
|---|---|---|
| 範圍 | 任意實數，可正可負 | 0 到 1 |
| 加總 | 沒有限制 | 加起來等於 1 |
| 怎麼得到機率 | 經過 **softmax** | — |

分數越高，代表模型認為這個字越可能是下一個字。

### 縮小版實測（numpy）

用 numpy 模擬同樣的運算，把維度縮小（6 個 token、隱藏維度 8、詞彙表 20），形狀的變化和真實模型一模一樣：

```
hidden 形狀:  (1, 6, 8)       ← 模擬 model.model 的輸出
logits 形狀:  (1, 6, 20)      ← 模擬 lm_head 的輸出（用矩陣乘法：8 維 → 20 個分數）
```

真實模型只是把 8 換成約 3072、把 20 換成約 3 萬。

In [ ]:
lm_model_output = model.lm_head(model_output[0])

## 步驟 6：取最後一個位置、選最高分、還原成文字

```python
token_id = lm_model_output[0,-1].argmax(dim=-1)
print(tokenizer.decode(token_id))
```

### 為什麼取 `[0, -1]`

`lm_model_output` 的形狀是 `(1, N, 詞彙表大小)`，所以：

| 索引 | 意思 |
|---|---|
| `[0` | 第 0 句（只有一句） |
| `, -1]` | **最後一個位置**（Python 的 `-1` 代表最後一個） |
| 結果形狀 | `(詞彙表大小,)`，一組分數 |

**為什麼是最後一個位置？** 因為模型在每個位置 t 預測的是「**第 t+1 個字**」：

```
位置：   The   capital   of   France   is
預測：   → capital  → of    → France   → is   → ???（我們要的下一個字）
```

前面幾個位置的預測是「句子裡已經有的字」（訓練時用來算誤差，推論時用不到）；**只有最後一個位置的預測，才是「這句話之後的下一個字」**。

縮小版實測（numpy）也印出每個位置的預測：

```
每個位置的預測 id: [19, 1, 12, 3, 15, 15]
                                      ↑ 最後一個才是我們要的下一個字
```

### `argmax(dim=-1)`

在詞彙表的所有分數中，**選出分數最高的那一個的位置（也就是 token id）**。

縮小版實測：

```
logits[0, -1] 形狀: (20,)
argmax（下一個 token 的 id）: 15
```

這就是**貪婪解碼**：永遠選機率最高的字。

### 變成機率之後，結果一樣

用 softmax 把分數轉成機率，**最大的位置不會變**（softmax 不改變大小順序），所以直接對 logits 做 `argmax` 就夠了，不用先算 softmax：

```
機率總和: 1.0    機率最大的 id: 15    與 logits 的 argmax 相同: True
前三名 (id, 機率): [(15, 0.654), (0, 0.212), (5, 0.051)]
```

（以上是縮小版 numpy 實測）想看模型有多「確定」，就要看機率而不是只看 argmax（見實驗區）。

### `tokenizer.decode(token_id)`

把 token id 還原成文字。預期印出 `Paris`（或帶一個前導空格的 ` Paris`）。我沒有實際執行，請以你跑出來的為準。

In [ ]:
token_id = lm_model_output[0,-1].argmax(dim=-1)  #取得最大機率的token id
print(tokenizer.decode(token_id))  #將token id轉換為文字

## 這一步與「生成一整段文字」的關係

你剛剛看到的，是生成的**一個迴圈**。完整的生成是這樣：

```
輸入：  The capital of France is
  第 1 輪：預測 → " Paris"     輸入變成：The capital of France is Paris
  第 2 輪：預測 → ","          輸入變成：The capital of France is Paris,
  第 3 輪：預測 → " and"       輸入變成：The capital of France is Paris, and
  ...重複，直到產生結束符號，或達到 max_new_tokens
```

| | 這份程式 | `model.generate` / `pipeline` |
|---|---|---|
| 預測幾次 | 1 次 | 重複很多次 |
| 內部做的事 | 本體 → lm_head → 取最後一個位置 → argmax | **完全一樣**，只是自動重複 |
| 額外的優化 | 沒有 | 有（例如快取已經算過的部分，避免重複計算） |

所以 `max_new_tokens=50` 的意思，就是「**把這份程式做的事重複最多 50 次**」。`do_sample=False` 就是每次都用 `argmax`。

## 如果不用 argmax 呢

`argmax` 永遠選最高分，結果固定。**取樣**則是依機率隨機抽：

| 方法 | 做法 | 結果 |
|---|---|---|
| 貪婪（`argmax`，這份程式） | 永遠選機率最高的 | 固定、可重現，但有時呆板重複 |
| 取樣 | 依機率隨機抽 | 每次不同，更有創意 |
| `temperature` | 在取樣前先調整機率的「尖銳度」 | 溫度低 → 更保守；溫度高 → 更隨機 |

這些策略，**全都是在「對 logits 怎麼選字」這一步做文章**。

---

# 重點整理

## 一、整份程式一句話

> 把提示詞分詞後，先通過 Transformer 本體得到每個位置的隱藏向量，再通過 `lm_head` 得到每個位置對詞彙表每個字的分數（logits），取最後一個位置、用 `argmax` 選分數最高的字，還原成文字，就是模型預測的下一個字。

## 二、整條資料流（必須搞懂）

| 步驟 | 程式 | 形狀 |
|---|---|---|
| 分詞 | `tokenizer(prompt, ...).input_ids` | `(1, N)` |
| ① 本體 | `model.model(input_ids=...)[0]` | `(1, N, 隱藏維度)` |
| ② 輸出頭 | `model.lm_head(...)` | `(1, N, 詞彙表大小)` |
| ③ 取最後位置 | `[0, -1]` | `(詞彙表大小,)` |
| ④ 選最高分 | `.argmax(dim=-1)` | 一個數字（token id） |
| ⑤ 還原 | `tokenizer.decode(...)` | 文字 |

## 三、必須搞懂的 10 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | 因果語言模型 | 根據前面所有的字，預測下一個字 |
| 2 | `model.model` | Transformer 本體，負責「理解」上下文 |
| 3 | 隱藏狀態 | 每個位置一個向量，吸收了它前面所有字的資訊 |
| 4 | `lm_head` | 線性層，把向量換算成「詞彙表每個字的分數」 |
| 5 | logits | `lm_head` 的輸出，未經 softmax 的原始分數 |
| 6 | softmax | 把 logits 轉成加總為 1 的機率，不改變大小順序 |
| 7 | 取最後一個位置 | 位置 t 預測第 t+1 個字，所以只有最後一個位置預測「下一個字」 |
| 8 | `argmax` | 選分數最高的 token id，也就是貪婪解碼 |
| 9 | token id | 詞彙表裡每個字的編號 |
| 10 | 生成 = 重複預測 | `generate` 就是把這一步重複很多次 |

## 四、最容易搞混的地方

1. **為什麼是 `model.model`？**
   外層的 `model` 是完整的 CausalLM，裡面有個屬性叫 `model`（Transformer 本體），另一個屬性叫 `lm_head`。

2. **`model_output[0]` 是什麼？**
   `model.model(...)` 回傳的物件中，第 0 個是最後一層的隱藏狀態。

3. **隱藏狀態和 logits 有什麼不同？**
   隱藏狀態是「理解之後的向量」；logits 是「對每個字的得分」。前者要經過 `lm_head` 才變成後者。

4. **為什麼只看最後一個位置？**
   因為只有它預測的是「這句話之後」的下一個字。

5. **logits 是機率嗎？**
   不是。要經過 softmax 才是機率。但因為 softmax 不改變順序，直接 `argmax` 就能得到同樣的答案。

6. **這份程式有用到 `generator` 嗎？**
   沒有，它是上一份留下來的，不影響結果。

7. **`import torch` 有用到嗎？**
   這份程式裡沒有直接用到，但它是模型運作的基礎，也可以用來寫 `torch.no_grad()`。

## 五、常見錯誤排查表

| 症狀 | 可能原因 | 處理 |
|---|---|---|
| `CUDA out of memory` | 模型約 7.6 GB（我估算），再加上沒關閉計算圖的記錄 | 用 `torch.no_grad()`；改用 4 位元量化載入；`device_map="auto"` |
| 輸入與模型不在同一個裝置的錯誤 | 張量沒搬到 GPU | 確認 `.to("cuda")`（或改用 `model.device`） |
| 把 `device_map` 改成 `"auto"` 後 `.to("cuda")` 出錯 | 模型可能被分散在多個裝置，輸入固定搬到 cuda 不一定對 | 改用 `model.device`，我沒有實測 |
| `AttributeError: ... has no attribute 'model'` | 換成別的模型後，本體的屬性名稱不同 | 不同模型的內部命名可能不同，用 `print(model)` 查看結構 |
| 印出的字不是 "Paris" | 模型預測不同、分詞或精度造成差異 | 看前幾名的機率（實驗 3），不一定是錯誤 |
| 載入時卡很久 | 第一次下載模型 | 等待，之後從快取讀取 |

## 六、觀察到的幾個地方（程式碼未修改，僅供參考）

1. **`generator` 建立了卻沒有使用**
   是上一份留下來的，不影響結果，但會讓人誤以為會用它生成。

2. **`import torch` 沒有用到**
   通常是為了寫 `torch.no_grad()`。

3. **沒有 `torch.no_grad()`**
   前向傳播會記錄計算圖，多佔顯存；這份只是推論，不需要。

4. **`.to("cuda")` 寫死裝置**
   和 `device_map="cuda"` 各寫一次。要換裝置時要改兩處，或改用 `model.device`。

5. **`model.model` 與 `model.lm_head` 是內部屬性**
   這是這個模型的內部結構，換成別的模型，命名可能不同，不是通用的 API。

6. **只看 `argmax`，看不到模型有多確定**
   同樣印出 "Paris"，機率可能是 0.95 也可能是 0.3，意義很不同。

7. **顯存**
   Phi-3-mini 以 16 位元載入，權重約 7.6 GB，8 GB 的顯卡可能放不下。

8. **`torch_dtype` 在新版可能改名為 `dtype`**。

## 七、可以自己做的實驗（需要時再自己加，不影響原程式）

```python
# 1. 看每一步的形狀
print(input_ids.shape)              # (1, N)
print(model_output[0].shape)        # (1, N, 隱藏維度)
print(lm_model_output.shape)        # (1, N, 詞彙表大小)

# 2. 用 no_grad 包起來，省記憶體
with torch.no_grad():
    model_output = model.model(input_ids=input_ids)
    lm_model_output = model.lm_head(model_output[0])

# 3. 看模型預測的前五名與機率（用 softmax 轉成機率）
probs = torch.softmax(lm_model_output[0, -1].float(), dim=-1)
top = torch.topk(probs, 5)
for p, i in zip(top.values, top.indices):
    print(repr(tokenizer.decode(i)), round(p.item(), 3))

# 4. 看「每個位置」的預測（位置 t 預測第 t+1 個字）
pred_ids = lm_model_output[0].argmax(dim=-1)
print(tokenizer.convert_ids_to_tokens(input_ids[0]))
print(tokenizer.convert_ids_to_tokens(pred_ids))

# 5. 自己寫一個生成迴圈（重複這份程式做的事）
ids = input_ids
with torch.no_grad():
    for _ in range(10):
        logits = model.lm_head(model.model(input_ids=ids)[0])
        next_id = logits[0, -1].argmax(dim=-1, keepdim=True).unsqueeze(0)
        ids = torch.cat([ids, next_id], dim=1)
print(tokenizer.decode(ids[0]))

# 6. 換成取樣：依機率隨機抽，而不是永遠取最高
next_id = torch.multinomial(probs, num_samples=1)

# 7. 加上溫度（溫度 < 1 更保守，> 1 更隨機）
temperature = 0.7
probs = torch.softmax(lm_model_output[0, -1].float() / temperature, dim=-1)

# 8. 換一個提示詞測試
# "The largest planet in our solar system is"

# 9. 看模型結構
print(model)
```

## 八、與前面學過的內容對照

| | 微調（分類、NER、QLoRA…） | `pipeline` 生成 | `model.generate` 手動拆解 | **這份：單步預測** |
|---|---|---|---|---|
| 層級 | 訓練 | 最高階 | 中階 | **最底層** |
| 看得到什麼 | loss、指標 | 最終文字 | token id 與文字 | **隱藏狀態、logits** |
| 目的 | 更新權重 | 使用 | 理解流程 | **理解模型內部** |

訓練時也用到這裡的 logits：對**每一個位置**的 logits，和「真正的下一個字」計算交叉熵損失。推論時只需要**最後一個位置**。

## 九、一句話記住

> **語言模型預測下一個字 = 本體理解上下文（隱藏狀態）→ `lm_head` 給每個字打分（logits）→ 取最後一個位置 → 選最高分。** 生成一整段文字，只是把這一步重複很多次。